# Glaucoma Dataset EDA Workflow

This notebook provides a reproducible workflow to analyze the key characteristics of the glaucoma datasets in this project.

Outputs include:
- label and split audits
- class balance summaries
- image quality proxies (brightness, contrast, blur)
- duplicate detection
- exportable report tables

## 1. Set Up Notebook Environment

Import required libraries, configure plotting style, and print key package versions for reproducibility.

In [3]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


pandas: 3.0.1
numpy: 2.4.2


## 2. Define Configuration and Paths

Set project root, input label CSVs, and runtime options for analysis.

In [ ]:
ROOT = Path('/raid/home/students/hawky_luc/P-le-Projet-Zenkolab')
OUTPUT_DIR = ROOT / 'outputs' / 'eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_IMAGES_FOR_STATS = None  # set an int (e.g., 500) to speed up testing
RANDOM_SEED = 42

CSV_CONFIG = [
    {
        'name': 'REFUGE2',
        'csv': ROOT / 'datasets/victorlemosml__refuge2/REFUGE2/labels.csv',
        'base_dir': ROOT / 'datasets/victorlemosml__refuge2/REFUGE2',
    },
    {
        'name': 'FUNDUS_SORTED',
        'csv': ROOT / 'datasets/sshikamaru__glaucoma-detection/Fundus_Train_Val_Data/Fundus_Scanes_Sorted/labels.csv',
        'base_dir': ROOT / 'datasets/sshikamaru__glaucoma-detection/Fundus_Train_Val_Data/Fundus_Scanes_Sorted',
    },
    {
        'name': 'ACRIMA',
        'csv': ROOT / 'datasets/sshikamaru__glaucoma-detection/ACRIMA/labels.csv',
        'base_dir': ROOT / 'datasets/sshikamaru__glaucoma-detection/ACRIMA',
    },
    {
        'name': 'ORIGA',
        'csv': ROOT / 'datasets/sshikamaru__glaucoma-detection/ORIGA/ORIGA/labels.csv',
        'base_dir': ROOT / 'datasets/sshikamaru__glaucoma-detection/ORIGA/ORIGA',
    },
]

for cfg in CSV_CONFIG:
    print(cfg['name'], '->', cfg['csv'])

## 3. Load or Create Sample Data

Load label CSVs from all datasets and harmonize schema into a single table for analysis.

In [ ]:
def load_one_dataset(cfg):
    df = pd.read_csv(cfg['csv'])
    df['dataset'] = cfg['name']

    if 'split' not in df.columns:
        df['split'] = 'all'
    if 'filename' not in df.columns and 'image_path' in df.columns:
        df['filename'] = df['image_path'].map(lambda p: Path(str(p)).name)

    if 'label' in df.columns:
        df['label'] = pd.to_numeric(df['label'], errors='coerce')
    else:
        df['label'] = np.nan

    df['has_label'] = df['label'].isin([0, 1])
    df['abs_image_path'] = df['image_path'].map(lambda p: cfg['base_dir'] / str(p))
    df['image_exists'] = df['abs_image_path'].map(lambda p: p.exists())
    return df

all_df = pd.concat([load_one_dataset(cfg) for cfg in CSV_CONFIG], ignore_index=True)

if MAX_IMAGES_FOR_STATS is not None and len(all_df) > MAX_IMAGES_FOR_STATS:
    all_df = all_df.sample(MAX_IMAGES_FOR_STATS, random_state=RANDOM_SEED).reset_index(drop=True)

print('Total rows:', len(all_df))
all_df.head()

## 4. Implement Core Processing Logic

Compute audits, image-level quality features, visualizations, and duplicate checks.

In [ ]:
audit_summary = (
    all_df.groupby(['dataset', 'split'], dropna=False)
    .agg(
        n_rows=('filename', 'count'),
        n_labeled=('has_label', 'sum'),
        n_missing_label=('has_label', lambda s: (~s).sum()),
        n_missing_image=('image_exists', lambda s: (~s).sum()),
        n_pos=('label', lambda s: (s == 1).sum()),
        n_neg=('label', lambda s: (s == 0).sum()),
    )
    .reset_index()
)
audit_summary['pos_ratio'] = (
    audit_summary['n_pos'] / (audit_summary['n_pos'] + audit_summary['n_neg']).replace(0, np.nan)
)

audit_summary.sort_values(['dataset', 'split'])

In [ ]:
plot_df = all_df[all_df['label'].isin([0, 1])].copy()
plot_df['label_name'] = plot_df['label'].map({0: 'non_glaucoma', 1: 'glaucoma'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=plot_df, x='dataset', hue='label_name', ax=axes[0])
axes[0].set_title('Class Counts by Dataset')
axes[0].tick_params(axis='x', rotation=20)

ratio_df = (
    plot_df.groupby(['dataset', 'label_name']).size()
    .groupby(level=0)
    .apply(lambda s: s / s.sum())
    .rename('ratio')
    .reset_index()
)
sns.barplot(data=ratio_df, x='dataset', y='ratio', hue='label_name', ax=axes[1])
axes[1].set_title('Class Ratios by Dataset')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
def image_stats(path):
    try:
        img = cv2.imread(str(path))
        if img is None:
            return None
        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return {
            'width': w,
            'height': h,
            'aspect_ratio': w / h if h else np.nan,
            'brightness': float(gray.mean()),
            'contrast': float(gray.std()),
            'blur_var': float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        }
    except (UnidentifiedImageError, OSError, ValueError):
        return None

stats_rows = []
for _, row in tqdm(all_df[all_df['image_exists']].iterrows(), total=int(all_df['image_exists'].sum())):
    s = image_stats(row['abs_image_path'])
    if s is None:
        continue
    stats_rows.append({
        'dataset': row['dataset'],
        'split': row['split'],
        'filename': row['filename'],
        'label': row['label'],
        **s,
    })

img_stats = pd.DataFrame(stats_rows)
print('Computed stats for images:', len(img_stats))
img_stats.head()

In [ ]:
if not img_stats.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    sns.boxplot(data=img_stats, x='dataset', y='width', ax=axes[0, 0])
    axes[0, 0].set_title('Width by Dataset')
    axes[0, 0].tick_params(axis='x', rotation=20)

    sns.boxplot(data=img_stats, x='dataset', y='height', ax=axes[0, 1])
    axes[0, 1].set_title('Height by Dataset')
    axes[0, 1].tick_params(axis='x', rotation=20)

    sns.boxplot(data=img_stats, x='dataset', y='brightness', ax=axes[1, 0])
    axes[1, 0].set_title('Brightness by Dataset')
    axes[1, 0].tick_params(axis='x', rotation=20)

    sns.boxplot(data=img_stats, x='dataset', y='blur_var', ax=axes[1, 1])
    axes[1, 1].set_title('Blur Proxy (Laplacian Variance)')
    axes[1, 1].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()
else:
    print('img_stats is empty. Skipping quality plots.')

In [ ]:
def file_md5(path, chunk_size=8192):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

hash_df = all_df[all_df['image_exists']][['dataset', 'split', 'filename', 'abs_image_path']].copy()
hash_df['md5'] = [file_md5(p) for p in tqdm(hash_df['abs_image_path'], total=len(hash_df))]

duplicates = hash_df[hash_df.duplicated('md5', keep=False)].sort_values('md5')
print('Duplicate-image rows:', len(duplicates))
duplicates.head(20)

## 5. Run Validation Checks

Run lightweight checks for schema consistency, labels, and key numeric ranges.

In [ ]:
required_cols = {'dataset', 'split', 'image_path', 'filename', 'label', 'abs_image_path', 'image_exists'}
missing_cols = required_cols - set(all_df.columns)
assert not missing_cols, f'Missing required columns: {missing_cols}'

valid_labels = all_df['label'].dropna().isin([0, 1]).all()
assert valid_labels, 'Found labels outside {0,1} after coercion.'

if not img_stats.empty:
    assert (img_stats['width'] > 0).all(), 'Non-positive image width found.'
    assert (img_stats['height'] > 0).all(), 'Non-positive image height found.'
    assert img_stats['brightness'].between(0, 255).all(), 'Brightness out of [0,255].'

print('All validation checks passed.')

## 6. Export Results

Save key summary tables for reuse in training and reporting.

In [ ]:
report = (
    img_stats.groupby('dataset')
    .agg(
        n_images=('filename', 'count'),
        width_median=('width', 'median'),
        height_median=('height', 'median'),
        brightness_median=('brightness', 'median'),
        contrast_median=('contrast', 'median'),
        blur_median=('blur_var', 'median'),
    )
    .join(
        all_df[all_df['label'].isin([0, 1])]
        .groupby('dataset')['label']
        .agg(
            n_pos=lambda s: (s == 1).sum(),
            n_neg=lambda s: (s == 0).sum(),
        )
    )
)
report['pos_ratio'] = report['n_pos'] / (report['n_pos'] + report['n_neg'])

report_path = OUTPUT_DIR / 'dataset_report.csv'
audit_path = OUTPUT_DIR / 'audit_summary.csv'
dups_path = OUTPUT_DIR / 'duplicate_rows.csv'

report.to_csv(report_path)
audit_summary.to_csv(audit_path, index=False)
duplicates.to_csv(dups_path, index=False)

print('Saved:', report_path)
print('Saved:', audit_path)
print('Saved:', dups_path)

report.sort_values('n_images', ascending=False)

In [ ]:
def show_samples(df, dataset, label, n=6):
    subset = df[(df['dataset'] == dataset) & (df['label'] == label) & (df['image_exists'])]
    if subset.empty:
        print(f'No samples for dataset={dataset}, label={label}')
        return

    sampled = subset.sample(min(n, len(subset)), random_state=RANDOM_SEED)
    cols = min(3, len(sampled))
    rows = int(np.ceil(len(sampled) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sampled.iterrows()):
        img = Image.open(row['abs_image_path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(f"{row['dataset']} | {row['filename']}")
        ax.axis('off')

    for ax in axes[len(sampled):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# Example: visualize one class from one dataset
show_samples(all_df, dataset='ORIGA', label=1, n=6)